# For all detected peaks, mark overlap with any other previous PAS annotation 

In [ ]:
import subprocess as sp
import pathlib
from collections import Counter
import concurrent.futures

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn import metrics

In [ ]:
base_path = pathlib.Path('/sc/arion/projects/CommonMind/yeon/p/APA/')

In [ ]:
import sys
sys.path.append(str(base_path))
from helpers import bed_modifiers

In [ ]:
# Parameters for intersecting with databases
INTERSECT_WINDOW = (50, 25)

INTERSECT_DIR = pathlib.Path(f'Bed_intersect_Up{INTERSECT_WINDOW[0]}bp_Down{INTERSECT_WINDOW[1]}bp')
INTERSECT_DIR.mkdir(exist_ok=True)
INTERSECT_DIR

# Define annotation paths

In [ ]:
ANNO_BED_FILES = {
    'GENCODE': base_path / 'ref/GENCODE.v38.ensembl.104.nochr/gencode.v38.polyAs.polyA_site.nochr.bed.gz',
    'PolyA_DB3': base_path / 'ref/polyADB3/polyADB_hg19.human.PAS.nochr.hg38.bed.gz',
    'PolyA-Seq': base_path / 'ref/polyA-Seq/polyA-Seq_merged_hg38.nochr.bed.gz',
    'PolyASite2': base_path / 'ref/polyASite/atlas.clusters.2.0.GRCh38.96.bed.gz',
    'PolyASite3': base_path / 'ref/polyASite/atlas.clusters.3.0.GRCh38.GENCODE_42.Blood_Brain_RPM0.1.nochr.bed.gz',
}
    


# Find all PAS groups

In [ ]:
PEAKS_PATH = base_path / 'run_SCAPTURE/1_make_pas/pl_ageXclass/peaks_5k-cells'

In [ ]:
groups = [p.name for p in PEAKS_PATH.glob('*/') if not p.is_file()]
groups.remove('stats')
groups

# Test single group

In [ ]:
group = groups[1]

bed_eval_paths = [
    PEAKS_PATH / f'{group}/{group}.intronic.peaks.evaluated.bed',
    PEAKS_PATH / f'{group}/{group}.exonic.peaks.evaluated.bed',
    PEAKS_PATH / f'{group}/{group}.3primeExtended.peaks.evaluated.bed',
]

dfs = []
for path in bed_eval_paths:
    df_each = pd.read_table(path, header=None, dtype={0: 'category'})
    del df_each[14]
    dfs.append(df_each)

df_eval = pd.concat(dfs)
df_eval

In [ ]:
df_eval_reduced = bed_modifiers.reduce_bed12_dataframe_5p_tosize(df_eval, INTERSECT_WINDOW[0], rename_chr=False)
df_eval_reduced_downadded = bed_modifiers.add_down_n_to_bed(df_eval_reduced, INTERSECT_WINDOW[1])
df_eval_reduced_downadded[13] = df_eval[13]

eval_bed_path = INTERSECT_DIR / f'{group}.eval_red.bed.gz'
df_eval_reduced_downadded.to_csv(eval_bed_path, compression='gzip', sep='\t', header=None, index=False)

In [ ]:
command = 'ml bedtools/2.31.0;' 
# Warning: don not run bedtools with -f and -split after version 2.29. there's a bug and it will not behave as expected.
# In below case, -f is not used so it is better to use 2.31.0. After 2.29, they fixed memory leak issue so it will run much faster.
for anno_name, anno_path in ANNO_BED_FILES.items():
    intersect_path = INTERSECT_DIR / f'{group}.eval_red.{anno_name}.intersect.gz'
    command += f' bedtools intersect -a {eval_bed_path} -b {anno_path} -s -split -u | gzip -> {intersect_path} ; '
sp.check_call(command, shell=True)

In [ ]:
dfs = []
for anno_name, anno_path in ANNO_BED_FILES.items():
    intersect_path = INTERSECT_DIR / f'{group}.eval_red.{anno_name}.intersect.gz'
    df_each = pd.read_table(intersect_path, header=None, dtype={0: 'category'})
    df_each[anno_name] = True
    dfs.append(df_each)
df_concat = pd.concat(dfs)
for anno_name in ANNO_BED_FILES:
    df_concat[anno_name] = df_concat[anno_name].fillna(0).astype('bool')
df_concat

In [ ]:
# Map to original bed
df_intersected = pd.merge(df_eval[range(0, 14)],
                        df_concat.groupby(3)[list(ANNO_BED_FILES.keys())].any(),
                        left_on=3, right_index=True, how='left').reset_index()
for anno_name in ANNO_BED_FILES:
    df_intersected[anno_name] = df_intersected[anno_name].fillna(0).astype('bool')
df_intersected['DeepPASS'] = df_intersected[13]=='positive'
df_intersected['pas_type'] = df_intersected[3].apply(lambda x: x.split('|')[-1])
df_intersected['pas_from'] = group

print(df_intersected[list(ANNO_BED_FILES.keys()) + ['DeepPASS']].sum())
print(Counter(df_intersected['pas_type']))

In [ ]:
df_intersected

# For all groups

In [ ]:
def run_for_group(grp):
    # Read original PASs
    bed_eval_paths = [
        PEAKS_PATH / f'{grp}/{grp}.intronic.peaks.evaluated.bed',
        PEAKS_PATH / f'{grp}/{grp}.exonic.peaks.evaluated.bed',
        PEAKS_PATH / f'{grp}/{grp}.3primeExtended.peaks.evaluated.bed',
    ]
    dfs = []
    for path in bed_eval_paths:
        df_each = pd.read_table(path, header=None, dtype={0: 'category'})
        del df_each[14]
        dfs.append(df_each)
    df_eval = pd.concat(dfs)

    # Reduce size, add downstream, and save
    df_eval_reduced = bed_modifiers.reduce_bed12_dataframe_5p_tosize(df_eval, INTERSECT_WINDOW[0], rename_chr=False)
    df_eval_reduced_downadded = bed_modifiers.add_down_n_to_bed(df_eval_reduced, INTERSECT_WINDOW[1])
    
    eval_bed_path = INTERSECT_DIR / f'{grp}.eval_red.bed.gz'
    df_eval_reduced_downadded.to_csv(eval_bed_path, compression='gzip', sep='\t', header=None, index=False)
    print(f'processed and saved {grp}, num PAS is: ', len(df_eval))

    # Intersect to annotations
    command = 'ml bedtools/2.31.0;' 
    for anno_name, anno_path in ANNO_BED_FILES.items():
        intersect_path = INTERSECT_DIR / f'{grp}.eval_red.{anno_name}.intersect.gz'
        command += f' bedtools intersect -a {eval_bed_path} -b {anno_path} -s -split -u | gzip -> {intersect_path} ; '
    sp.run(command, shell=True, stderr=sp.DEVNULL)

    # Read intersect again
    dfs = []
    for anno_name, anno_path in ANNO_BED_FILES.items():
        intersect_path = INTERSECT_DIR / f'{grp}.eval_red.{anno_name}.intersect.gz'
        df_each = pd.read_table(intersect_path, header=None, dtype={0: 'category'})
        df_each[anno_name] = True
        dfs.append(df_each)
    df_concat = pd.concat(dfs)
    for anno_name in ANNO_BED_FILES:
        df_concat[anno_name] = df_concat[anno_name].fillna(0).astype('bool')

    # Map to original bed
    df_intersected = pd.merge(df_eval[range(0, 14)],
                              df_concat.groupby(3)[list(ANNO_BED_FILES.keys())].any(),
                              left_on=3, right_index=True, how='left').reset_index()
    for anno_name in ANNO_BED_FILES:
        df_intersected[anno_name] = df_intersected[anno_name].fillna(0).astype('bool')
    df_intersected['DeepPASS'] = df_intersected[13]=='positive'
    df_intersected['pas_type'] = df_intersected[3].apply(lambda x: x.split('|')[-1])
    df_intersected['pas_from'] = grp
    
    return df_intersected




In [ ]:
dfs = []  
with concurrent.futures.ProcessPoolExecutor(max_workers=40) as executor:
    future_to_group = {executor.submit(run_for_group, group): group for group in groups}
    
    for future in concurrent.futures.as_completed(future_to_group):
        group = future_to_group[future]  # Identify the corresponding group
        try:
            dfs.append(future.result())  
        except Exception as exc:
            print(f"Group {group} generated an exception: {exc}")

df_int_all = pd.concat(dfs, ignore_index=True)
df_int_all['# Anno (including GENCODE)'] = df_int_all['GENCODE PolyA_DB3 PolyA-Seq PolyASite2 PolyASite3'.split()].sum(axis=1)
df_int_all['# Anno (without GENCODE)'] = df_int_all['PolyA_DB3 PolyA-Seq PolyASite2 PolyASite3'.split()].sum(axis=1)
df_int_all


In [ ]:
df_int_all.to_pickle(INTERSECT_DIR / 'all_peaks_evaluated_intersected.pkl')
df_int_all.to_csv(INTERSECT_DIR / 'all_peaks_evaluated_intersected.tsv.gz', sep='\t', compression='gzip')

In [ ]:
#~df_int_all['GENCODE']]

df_gencode = df_int_all.groupby('pas_from')[['GENCODE']].sum()

se_not_gencode = ~df_int_all['GENCODE']
se_more1_anno = (df_int_all[['PolyA_DB3', 'PolyA-Seq', 'PolyASite2', 'PolyASite3']].sum(axis=1) >= 2)

df_gencode['Anno >= 2'] = df_int_all[se_not_gencode & se_more1_anno].groupby('pas_from')[0].count()


fig, ax = plt.subplots(figsize=(8, 3))
df_gencode.plot(kind='bar', stacked=True, ax=ax)
ax.set_xticklabels(ax.get_xticklabels(), rotation=90, )
plt.show()

In [ ]:
df_int_all['GENCODE PolyA_DB3 PolyA-Seq PolyASite2 PolyASite3 DeepPASS pas_type pas_from'.split()]

In [ ]:

df_num_anno = df_int_all.groupby('pas_from')['# Anno (including GENCODE)'].apply(Counter).reset_index().pivot(
    index = 'pas_from',
    columns = 'level_1',
    values = '# Anno (including GENCODE)'
)

fig, ax = plt.subplots(figsize=(8, 4))
df_num_anno.plot(kind='bar', stacked=True, ax=ax)
ax.set_xticklabels(ax.get_xticklabels(), rotation=90, )
ax.legend(loc = (1.01, 0.0), title='# Matched annotations\n(including GENCODE)')
ax.set_title('All peaks')
plt.show()

In [ ]:
df_num_anno = df_int_all[df_int_all['DeepPASS']].groupby('pas_from')['# Anno (including GENCODE)'].apply(Counter).reset_index().pivot(
    index = 'pas_from',
    columns = 'level_1',
    values = '# Anno (including GENCODE)'
)

fig, ax = plt.subplots(figsize=(8, 4))
df_num_anno.plot(kind='bar', stacked=True, ax=ax)
ax.set_xticklabels(ax.get_xticklabels(), rotation=90, )
ax.legend(loc = (1.01, 0.0), title='# Matched annotations\n(including GENCODE)')
ax.set_title('DeepPASS (SCAPTURE) positive')
plt.show()

In [ ]:
# Add non-intersected, draw upset plot?

from upsetplot import UpSet, from_contents


In [ ]:
sample = 'Adolescence-Oligo'
df_cut = df_int_all[df_int_all['pas_from']==sample].copy()

annos_tested = 'GENCODE PolyA_DB3 PolyA-Seq PolyASite2 PolyASite3 DeepPASS'.split()
cat_elements = {an: set(df_cut[df_cut[an]][3]) for an in annos_tested}

upset_data = from_contents(cat_elements)
upset = UpSet(
    upset_data,
    show_counts="%d",
    show_percentages=True,
    sort_by="cardinality",
    #sort_by="degree",          
    min_subset_size=500
)

fig = plt.figure(figsize=(10, 6))
upset.plot(fig=fig)

fig.suptitle(f"SCAPTURE DeepPASS PASs from {sample}")
plt.savefig(f"SCAPTURE_DeepPASS_PASs_from_{sample}_overlap_with_databases.png", bbox_inches='tight')
plt.show()